# BioJEPA v0.6 Data Prep - Notebook 2: Perturbation Embeddings

This notebook handles:
1. Extracting unique perturbations per dataset
2. Linking perturbations to sequences (sgRNA protospacer, SMILES)
3. Mapping perturbations to target proteins (ENSG -> UniProt -> sequence)
4. Running embedding models (NucleotideTransformer, ESM-2, ChemMRL)
5. Saving feature banks with index mappings

**Inputs (from Notebook 1):**
- `gene_to_idx.json` - ENSG -> index for gene universe
- `dataset_splits.json` - train/val/test perturbation sets per dataset

**Outputs:**
- `pert_embd/seq_banks/dna_embeddings.npy` - [N_dna, 1536]
- `pert_embd/seq_banks/dna_to_idx.json` - sgID_AB -> idx
- `pert_embd/seq_banks/chemical_embeddings.npy` - [N_chem, 1024]
- `pert_embd/seq_banks/chemical_to_idx.json` - SMILES -> idx
- `pert_embd/target_banks/protein_targets.npy` - [N_genes, 320]
- `pert_embd/target_banks/gene_to_idx.json` - ENSG -> idx
- `pert_embd/input_to_id.json` - "GENE_mode_dataset" -> seq_idx (for pathway evals)

In [ ]:
from pathlib import Path
from collections import defaultdict
from Bio import SeqIO
from tqdm import tqdm
import pandas as pd
import numpy as np
import scanpy as sc
import mygene
import torch
import json
import gzip
import gc

In [ ]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')

seq_banks_dir = data_dir / 'pert_embd' / 'seq_banks'
target_banks_dir = data_dir / 'pert_embd' / 'target_banks'
seq_banks_dir.mkdir(parents=True, exist_ok=True)
target_banks_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
with open(data_dir / 'gene_to_idx.json') as f:
    gene_to_idx = json.load(f)

with open(data_dir / 'dataset_splits.json') as f:
    dataset_splits = json.load(f)

print(f'Loaded gene universe with {len(gene_to_idx)} genes')

In [ ]:
def is_valid(val):
    if pd.isna(val):
        return False
    if isinstance(val, str) and (val.strip() == '' or val.strip().lower() == 'nan'):
        return False
    return True

## Step 1: Load CRISPRi Reference Library

Maps sgID -> protospacer sequence for k562e_raw, rep1e, k562gw

In [ ]:
crispr_df = pd.read_csv(ref_dir / 'crispr' / 'hcrispri_all.csv')
crispr_df['sgID_clean'] = crispr_df['sgID'].astype(str).str.replace(',', '-').str.strip()

sgid_to_seq = dict(zip(crispr_df['sgID_clean'], crispr_df['protospacer sequence'].str.strip()))
sgid_to_gene = dict(zip(crispr_df['sgID_clean'], crispr_df['gene'].str.strip()))

print(f'Loaded {len(sgid_to_seq)} sgID -> protospacer mappings')

## Step 2: Extract Unique sgID_AB from CRISPRi Datasets

k562e_raw, rep1e, k562gw all use dual-guide CRISPRi

In [ ]:
datasets = {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e': ref_dir / 'rep1e' / 'rpe1_raw_singlecell_01.h5ad',
    'k562gw': ref_dir / 'k562gw' / 'perturb_processed.h5ad',
}

In [ ]:
def get_sequences_from_sgid_ab(sgid_ab):
    if not is_valid(sgid_ab):
        return None, None
    sgid_ab = str(sgid_ab).replace(',', '-').strip()
    parts = sgid_ab.split('|')
    seq_a = sgid_to_seq.get(parts[0].strip())
    seq_b = sgid_to_seq.get(parts[1].strip()) if len(parts) > 1 else None
    return seq_a, seq_b

def get_gene_from_sgid_ab(sgid_ab):
    if not is_valid(sgid_ab):
        return None
    sgid_ab = str(sgid_ab).replace(',', '-').strip()
    parts = sgid_ab.split('|')
    return sgid_to_gene.get(parts[0].strip())

In [ ]:
crispri_sgids = {}  # dataset -> set of unique sgID_AB
sgid_ab_to_gene = {}  # sgID_AB -> target gene name

for ds_name, ds_path in datasets.items():
    print(f'Loading {ds_name}...')
    adata = sc.read_h5ad(ds_path, backed='r')
    
    if 'sgID_AB' in adata.obs.columns:
        sgids = set(adata.obs['sgID_AB'].dropna().unique())
    else:
        print(f'  No sgID_AB column, checking alternatives...')
        sgids = set()
    
    crispri_sgids[ds_name] = sgids
    print(f'  Found {len(sgids)} unique sgID_AB entries')
    
    for sgid in sgids:
        if sgid not in sgid_ab_to_gene:
            gene = get_gene_from_sgid_ab(sgid)
            if gene:
                sgid_ab_to_gene[sgid] = gene

In [ ]:
all_crispri_sgids = set()
for ds_sgids in crispri_sgids.values():
    all_crispri_sgids.update(ds_sgids)

print(f'Total unique sgID_AB across CRISPRi datasets: {len(all_crispri_sgids)}')

## Step 3: Load Adamson Protospacer Mapping

In [ ]:
adamson_df = pd.read_csv(ref_dir / 'adamson' / 'adamson_protospacer_sgrna.csv')
adamson_gene_to_protospacer = dict(zip(adamson_df['Gene'].str.strip(), adamson_df['Protospacer'].str.strip()))

print(f'Loaded {len(adamson_gene_to_protospacer)} Adamson gene -> protospacer mappings')

In [ ]:
adamson_path = ref_dir / 'adamson' / 'AdamsonWeissman2016_GSM2406681_10X010.h5ad'
adata = sc.read_h5ad(adamson_path, backed='r')

if 'perturbation' in adata.obs.columns:
    adamson_perts = set(adata.obs['perturbation'].dropna().unique())
elif 'condition' in adata.obs.columns:
    adamson_perts = set(adata.obs['condition'].dropna().unique())
else:
    adamson_perts = set()

adamson_perts = {p for p in adamson_perts if is_valid(p) and str(p).lower() not in ['control', 'ctrl', 'nan']}
print(f'Adamson unique perturbations: {len(adamson_perts)}')

## Step 4: Load Norman sgRNA and Target Mappings

In [ ]:
norman_sgrna_df = pd.read_csv(ref_dir / 'norman' / 'norman_sgrna.csv')
norman_target_df = pd.read_csv(ref_dir / 'norman' / 'norman_guideid_ensemble_id_map.csv')

norman_sgrna_df.head()

In [ ]:
norman_target_df.head()

In [ ]:
norman_gene_to_protospacer_a = dict(zip(norman_sgrna_df['gene_A'].str.strip(), norman_sgrna_df['protospacer_sequence_A'].str.strip()))
norman_gene_to_protospacer_b = {}
for _, row in norman_sgrna_df.iterrows():
    if is_valid(row['gene_B']) and str(row['gene_B']).lower() != 'negctrl0':
        norman_gene_to_protospacer_b[row['gene_B'].strip()] = row['protospacer_sequence_B'].strip()

norman_gene_to_ensg = {}
for _, row in norman_target_df.iterrows():
    if is_valid(row.get('first_target')) and is_valid(row.get('first_id')):
        norman_gene_to_ensg[row['first_target'].strip()] = row['first_id'].strip()
    if is_valid(row.get('second_target')) and is_valid(row.get('second_id')):
        norman_gene_to_ensg[row['second_target'].strip()] = row['second_id'].strip()

print(f'Norman gene -> protospacer_A: {len(norman_gene_to_protospacer_a)}')
print(f'Norman gene -> protospacer_B: {len(norman_gene_to_protospacer_b)}')
print(f'Norman gene -> ENSG: {len(norman_gene_to_ensg)}')

In [ ]:
norman_path = ref_dir / 'norman' / 'NormanWeissman2019_filtered.h5ad'
adata = sc.read_h5ad(norman_path, backed='r')

if 'guide_id' in adata.obs.columns:
    norman_guide_ids = set(adata.obs['guide_id'].dropna().unique())
else:
    norman_guide_ids = set()

norman_guide_ids = {g for g in norman_guide_ids if is_valid(g) and str(g).lower() not in ['control', 'ctrl', 'nan']}
print(f'Norman unique guide_ids: {len(norman_guide_ids)}')

## Step 5: Load Sciplex Chemical Perturbations

In [ ]:
sciplex_path = ref_dir / 'sciplex' / 'SrivatsanTrapnell2020_sciplex3.h5ad'
adata = sc.read_h5ad(sciplex_path, backed='r')

adata.obs.columns.tolist()

In [ ]:
if 'product_name' in adata.obs.columns:
    sciplex_drugs = set(adata.obs['product_name'].dropna().unique())
elif 'perturbation' in adata.obs.columns:
    sciplex_drugs = set(adata.obs['perturbation'].dropna().unique())
else:
    sciplex_drugs = set()

sciplex_drugs = {d for d in sciplex_drugs if is_valid(d) and str(d).lower() not in ['control', 'vehicle', 'dmso', 'nan']}
print(f'Sciplex unique drugs: {len(sciplex_drugs)}')

In [ ]:
list(sciplex_drugs)[:10]

## Step 6: Map Drugs to SMILES via ChEMBL

We use the chembl_36_chemreps.txt file which contains SMILES for ChEMBL compounds. We'll also use PubChemPy as a fallback.

In [ ]:
import pubchempy as pcp

In [ ]:
drug_to_smiles = {}

for drug in tqdm(sciplex_drugs, desc='Fetching SMILES from PubChem'):
    try:
        compounds = pcp.get_compounds(drug, 'name')
        if compounds:
            smiles = compounds[0].isomeric_smiles
            if smiles:
                drug_to_smiles[drug] = smiles
    except Exception as e:
        pass

print(f'Found SMILES for {len(drug_to_smiles)} / {len(sciplex_drugs)} drugs')

In [ ]:
with open(data_dir / 'pert_embd' / 'drug_to_smiles.json', 'w') as f:
    json.dump(drug_to_smiles, f, indent=2)

## Step 7: Load UniProt Protein Sequences

Map ENSG -> UniProt accession -> protein sequence

In [ ]:
uniprot_fasta = ref_dir / 'uniprot' / 'UP000005640_9606.fasta.gz'

acc_to_seq = {}
with gzip.open(uniprot_fasta, 'rt') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        parts = record.id.split('|')
        if len(parts) >= 2:
            acc_to_seq[parts[1]] = str(record.seq)

print(f'Loaded {len(acc_to_seq)} UniProt sequences')

In [ ]:
all_target_genes = set()

for sgid, gene in sgid_ab_to_gene.items():
    all_target_genes.add(gene)

for gene in adamson_perts:
    gene_clean = str(gene).split('/')[0].strip() if '/' in str(gene) else str(gene).strip()
    all_target_genes.add(gene_clean)

for gene in norman_gene_to_ensg.keys():
    all_target_genes.add(gene)

print(f'Total unique target genes: {len(all_target_genes)}')

In [ ]:
gene_to_ensg = {}
ensg_to_gene = {}

for ensg, gene in zip(gene_to_idx.keys(), gene_to_idx.keys()):
    if ensg.startswith('ENSG'):
        continue

for ensg in gene_to_idx.keys():
    if ensg.startswith('ENSG'):
        continue

with open(data_dir / 'gene_names.json') as f:
    gene_names = json.load(f)

for i, (ensg, idx) in enumerate(gene_to_idx.items()):
    if ensg.startswith('ENSG'):
        name = gene_names[idx] if idx < len(gene_names) else ensg
        gene_to_ensg[name] = ensg
        ensg_to_gene[ensg] = name

print(f'Gene -> ENSG mappings: {len(gene_to_ensg)}')

In [ ]:
target_ensg_ids = set()

for gene in all_target_genes:
    if gene in gene_to_ensg:
        target_ensg_ids.add(gene_to_ensg[gene])
    elif gene.startswith('ENSG'):
        target_ensg_ids.add(gene)

for ensg in norman_gene_to_ensg.values():
    if is_valid(ensg):
        target_ensg_ids.add(ensg)

print(f'Target ENSG IDs to embed: {len(target_ensg_ids)}')

In [ ]:
mg = mygene.MyGeneInfo()

results = mg.querymany(list(target_ensg_ids), scopes='ensembl.gene', fields='uniprot', species='human', verbose=False)

ensg_to_uniprot = {}
for res in results:
    if 'uniprot' in res:
        if 'Swiss-Prot' in res['uniprot']:
            val = res['uniprot']['Swiss-Prot']
            ensg_to_uniprot[res['query']] = val[0] if isinstance(val, list) else val

print(f'ENSG -> UniProt mappings: {len(ensg_to_uniprot)}')

In [ ]:
ensg_to_protein_seq = {}
missing_proteins = []

for ensg in target_ensg_ids:
    acc = ensg_to_uniprot.get(ensg)
    if acc and acc in acc_to_seq:
        ensg_to_protein_seq[ensg] = acc_to_seq[acc]
    else:
        missing_proteins.append(ensg)

print(f'Found protein sequences for {len(ensg_to_protein_seq)} / {len(target_ensg_ids)} targets')
print(f'Missing: {len(missing_proteins)}')

## Step 8: Generate DNA Embeddings (NucleotideTransformer)

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
nt_model_name = 'InstaDeepAI/NTv3_650M_pre'
nt_tokenizer = AutoTokenizer.from_pretrained(nt_model_name, trust_remote_code=True)
nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_name, trust_remote_code=True)
nt_model.to(device).eval()

print(f'Loaded NucleotideTransformer: {nt_model_name}')

In [ ]:
def get_dna_embedding(sequence):
    inputs = nt_tokenizer([sequence], return_tensors='pt', padding='max_length', max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    attention_mask = (inputs['input_ids'] != nt_tokenizer.pad_token_id).long()
    
    with torch.no_grad():
        outputs = nt_model(**inputs, output_hidden_states=True)
    
    embeddings = outputs.hidden_states[-1]
    mask = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
    pooled = (embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return pooled.squeeze(0).cpu().numpy()

In [ ]:
dna_vectors = []
dna_to_idx = {}
sgid_ab_to_dna_idx = {}

for sgid_ab in tqdm(all_crispri_sgids, desc='Embedding CRISPRi sgRNAs'):
    seq_a, seq_b = get_sequences_from_sgid_ab(sgid_ab)
    
    if seq_a is None:
        continue
    
    emb_a = get_dna_embedding(seq_a)
    
    if seq_b is not None:
        emb_b = get_dna_embedding(seq_b)
        combined = (emb_a + emb_b) / 2
    else:
        combined = emb_a
    
    idx = len(dna_vectors)
    dna_vectors.append(combined)
    dna_to_idx[sgid_ab] = idx
    sgid_ab_to_dna_idx[sgid_ab] = idx

print(f'Generated {len(dna_vectors)} CRISPRi DNA embeddings')

In [ ]:
adamson_gene_to_dna_idx = {}

for gene in tqdm(adamson_perts, desc='Embedding Adamson sgRNAs'):
    gene_clean = str(gene).split('/')[0].strip() if '/' in str(gene) else str(gene).strip()
    protospacer = adamson_gene_to_protospacer.get(gene_clean) or adamson_gene_to_protospacer.get(gene)
    
    if protospacer is None or not is_valid(protospacer):
        continue
    
    emb = get_dna_embedding(protospacer)
    idx = len(dna_vectors)
    dna_vectors.append(emb)
    adamson_gene_to_dna_idx[gene] = idx

print(f'Generated {len(adamson_gene_to_dna_idx)} Adamson DNA embeddings')
print(f'Total DNA embeddings: {len(dna_vectors)}')

In [ ]:
norman_guide_to_dna_idx = {}

for _, row in tqdm(norman_sgrna_df.iterrows(), total=len(norman_sgrna_df), desc='Embedding Norman sgRNAs'):
    gene_a = row['gene_A']
    gene_b = row['gene_B']
    
    if not is_valid(gene_a):
        continue
    
    guide_id = f"{gene_a}_{gene_b}" if is_valid(gene_b) and str(gene_b).lower() != 'negctrl0' else gene_a
    
    seq_a = row['protospacer_sequence_A']
    seq_b = row['protospacer_sequence_B'] if is_valid(row.get('protospacer_sequence_B')) else None
    
    if not is_valid(seq_a):
        continue
    
    emb_a = get_dna_embedding(seq_a)
    
    if seq_b is not None and is_valid(seq_b) and str(gene_b).lower() != 'negctrl0':
        emb_b = get_dna_embedding(seq_b)
        combined = (emb_a + emb_b) / 2
    else:
        combined = emb_a
    
    idx = len(dna_vectors)
    dna_vectors.append(combined)
    norman_guide_to_dna_idx[guide_id] = idx

print(f'Generated {len(norman_guide_to_dna_idx)} Norman DNA embeddings')
print(f'Total DNA embeddings: {len(dna_vectors)}')

In [ ]:
del nt_model, nt_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
dna_embeddings = np.stack(dna_vectors, axis=0).astype(np.float32)
print(f'DNA embeddings shape: {dna_embeddings.shape}')

np.save(seq_banks_dir / 'dna_embeddings.npy', dna_embeddings)

with open(seq_banks_dir / 'dna_to_idx.json', 'w') as f:
    json.dump(dna_to_idx, f)

with open(seq_banks_dir / 'adamson_gene_to_dna_idx.json', 'w') as f:
    json.dump(adamson_gene_to_dna_idx, f)

with open(seq_banks_dir / 'norman_guide_to_dna_idx.json', 'w') as f:
    json.dump(norman_guide_to_dna_idx, f)

## Step 9: Generate Protein Target Embeddings (ESM-2)

In [ ]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
esm_model_name = 'facebook/esm2_t6_8M_UR50D'
esm_tokenizer = AutoTokenizer.from_pretrained(esm_model_name)
esm_model = AutoModel.from_pretrained(esm_model_name)
esm_model.to(device).eval()

print(f'Loaded ESM-2: {esm_model_name}')

In [ ]:
def get_protein_embedding(protein_seq):
    inputs = esm_tokenizer(protein_seq, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = esm_model(**inputs)
    
    emb = outputs.last_hidden_state[0].mean(dim=0)
    return emb.cpu().numpy()

In [ ]:
protein_vectors = []
ensg_to_target_idx = {}

for ensg, protein_seq in tqdm(ensg_to_protein_seq.items(), desc='Embedding target proteins'):
    emb = get_protein_embedding(protein_seq)
    idx = len(protein_vectors)
    protein_vectors.append(emb)
    ensg_to_target_idx[ensg] = idx

print(f'Generated {len(protein_vectors)} protein target embeddings')

In [ ]:
del esm_model, esm_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
protein_embeddings = np.stack(protein_vectors, axis=0).astype(np.float32)
print(f'Protein embeddings shape: {protein_embeddings.shape}')

np.save(target_banks_dir / 'protein_targets.npy', protein_embeddings)

with open(target_banks_dir / 'gene_to_idx.json', 'w') as f:
    json.dump(ensg_to_target_idx, f)

## Step 10: Generate Chemical Embeddings (ChemMRL)

In [ ]:
chem_model_name = 'Derify/ChemMRL'
chem_tokenizer = AutoTokenizer.from_pretrained(chem_model_name)
chem_model = AutoModel.from_pretrained(chem_model_name)
chem_model.to(device).eval()

print(f'Loaded ChemMRL: {chem_model_name}')

In [ ]:
def get_chemical_embedding(smiles):
    inputs = chem_tokenizer(smiles, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = chem_model(**inputs)
    
    emb = outputs.last_hidden_state[0].mean(dim=0)
    return emb.cpu().numpy()

In [ ]:
chemical_vectors = []
smiles_to_idx = {}
drug_to_chem_idx = {}

for drug, smiles in tqdm(drug_to_smiles.items(), desc='Embedding chemicals'):
    if smiles in smiles_to_idx:
        drug_to_chem_idx[drug] = smiles_to_idx[smiles]
        continue
    
    emb = get_chemical_embedding(smiles)
    idx = len(chemical_vectors)
    chemical_vectors.append(emb)
    smiles_to_idx[smiles] = idx
    drug_to_chem_idx[drug] = idx

print(f'Generated {len(chemical_vectors)} chemical embeddings')

In [ ]:
del chem_model, chem_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
if chemical_vectors:
    chemical_embeddings = np.stack(chemical_vectors, axis=0).astype(np.float32)
    print(f'Chemical embeddings shape: {chemical_embeddings.shape}')
    
    np.save(seq_banks_dir / 'chemical_embeddings.npy', chemical_embeddings)
    
    with open(seq_banks_dir / 'chemical_to_idx.json', 'w') as f:
        json.dump(smiles_to_idx, f)
    
    with open(seq_banks_dir / 'drug_to_chem_idx.json', 'w') as f:
        json.dump(drug_to_chem_idx, f)
else:
    print('No chemical embeddings generated')

## Step 11: Build input_to_id.json for Pathway Evals

Maps "GENE_mode_dataset" -> seq_idx for pathway analysis

In [ ]:
input_to_id = {}

for sgid_ab, idx in sgid_ab_to_dna_idx.items():
    gene = sgid_ab_to_gene.get(sgid_ab)
    if gene:
        for ds in ['k562e', 'rep1e', 'k562gw']:
            key = f'{gene}_crispri_{ds}'
            if key not in input_to_id:
                input_to_id[key] = idx

for gene, idx in adamson_gene_to_dna_idx.items():
    gene_clean = str(gene).split('/')[0].strip() if '/' in str(gene) else str(gene).strip()
    key = f'{gene_clean}_crispri_adamson'
    input_to_id[key] = idx

for guide_id, idx in norman_guide_to_dna_idx.items():
    key = f'{guide_id}_crispra_norman'
    input_to_id[key] = idx

for drug, idx in drug_to_chem_idx.items():
    key = f'{drug}_inhibitor_sciplex'
    input_to_id[key] = idx

print(f'Built input_to_id with {len(input_to_id)} entries')

In [ ]:
with open(data_dir / 'pert_embd' / 'input_to_id.json', 'w') as f:
    json.dump(input_to_id, f)

## Step 12: Save Gene-to-Target Mappings for Training

In [ ]:
gene_to_target_idx = {}

for gene, ensg in gene_to_ensg.items():
    if ensg in ensg_to_target_idx:
        gene_to_target_idx[gene] = ensg_to_target_idx[ensg]

for ensg, idx in ensg_to_target_idx.items():
    gene_to_target_idx[ensg] = idx

for gene, ensg in norman_gene_to_ensg.items():
    if ensg in ensg_to_target_idx:
        gene_to_target_idx[gene] = ensg_to_target_idx[ensg]

print(f'Gene to target idx: {len(gene_to_target_idx)} mappings')

In [ ]:
with open(target_banks_dir / 'gene_to_target_idx.json', 'w') as f:
    json.dump(gene_to_target_idx, f)

## Summary

In [ ]:
print('=== Data Prep Notebook 2 Complete ===')
print(f'\nDNA Embeddings: {dna_embeddings.shape}')
print(f'  - CRISPRi sgID_AB: {len(sgid_ab_to_dna_idx)}')
print(f'  - Adamson genes: {len(adamson_gene_to_dna_idx)}')
print(f'  - Norman guides: {len(norman_guide_to_dna_idx)}')
print(f'\nProtein Embeddings: {protein_embeddings.shape}')
print(f'  - Target genes: {len(ensg_to_target_idx)}')
if chemical_vectors:
    print(f'\nChemical Embeddings: {chemical_embeddings.shape}')
    print(f'  - Unique SMILES: {len(smiles_to_idx)}')
    print(f'  - Drugs: {len(drug_to_chem_idx)}')
print(f'\ninput_to_id entries: {len(input_to_id)}')

In [ ]:
print('\nOutput files:')
for f in seq_banks_dir.glob('*'):
    print(f'  {f.name}')
for f in target_banks_dir.glob('*'):
    print(f'  {f.name}')